In [ ]:
# Jupyter Notebook 数据探索
# 保存为 notebooks/01_data_exploration.ipynb

# %% [markdown]
# # 学生学业数据探索分析
#
# 本笔记本用于探索和分析学生学业数据

# %% [markdown]
# ## 1. 导入库

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# 导入display函数（在Jupyter中自动可用，这里显式导入以确保兼容性）
try:
    from IPython.display import display, Markdown
    IN_JUPYTER = True
except ImportError:
    # 如果不在Jupyter环境中，定义一个简单的display函数
    def display(obj, *args, **kwargs):
        if hasattr(obj, 'head'):
            print(obj.head())
        elif isinstance(obj, pd.DataFrame):
            print(obj.head())
        elif isinstance(obj, pd.Series):
            print(obj.head())
        else:
            print(obj)

    def Markdown(text):
        print(text)

    IN_JUPYTER = False
    print("⚠️  不在Jupyter环境中，使用简单显示模式")

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

# 添加src目录到路径
sys.path.append('../src')

# %% [markdown]
# ## 2. 加载数据

# %%
# 加载数据
data_path = "../data/DATA (1).csv"

try:
    df = pd.read_csv(data_path)
    print(f"✅ 数据加载成功！")
    print(f"📊 数据形状: {df.shape}")
    print(f"📝 数据列名: {list(df.columns)}")

    # 显示前几行数据
    print("\n📋 数据预览（前5行）:")
    display(df.head())

except FileNotFoundError:
    print(f"❌ 文件未找到: {data_path}")
    print("请确保数据文件在正确的位置")
    df = None
except Exception as e:
    print(f"❌ 加载数据时出错: {str(e)}")
    df = None

# %% [markdown]
# ## 3. 数据概览

# %%
if df is not None:
    # 基本信息
    print("\n📊 数据基本信息:")
    print(f"数据集形状: {df.shape}")
    print(f"行数: {df.shape[0]}")
    print(f"列数: {df.shape[1]}")

    # 数据类型
    print("\n📝 数据类型分布:")
    dtype_counts = df.dtypes.value_counts()
    for dtype, count in dtype_counts.items():
        print(f"  {dtype}: {count}列")

# %% [markdown]
# ## 4. 缺失值分析

# %%
if df is not None:
    # 缺失值统计
    missing = df.isnull().sum()
    missing_percent = (missing / len(df)) * 100

    missing_df = pd.DataFrame({
        '缺失数量': missing,
        '缺失比例(%)': missing_percent
    })

    missing_df = missing_df[missing_df['缺失数量'] > 0].sort_values('缺失数量', ascending=False)

    if len(missing_df) > 0:
        print("⚠️  发现缺失值的列:")
        if IN_JUPYTER:
            display(missing_df)
        else:
            print(missing_df.to_string())

        # 可视化缺失值
        plt.figure(figsize=(12, 6))
        missing_df['缺失比例(%)'].plot(kind='bar')
        plt.title('缺失值比例')
        plt.ylabel('缺失比例(%)')
        plt.xlabel('列名')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("✅ 没有缺失值！")

# %% [markdown]
# ## 5. 数值型特征分析

# %%
if df is not None:
    # 提取数值型列
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    if len(numeric_cols) > 0:
        print(f"🔢 找到 {len(numeric_cols)} 个数值型特征")

        # 描述性统计
        print("\n📈 数值型特征描述性统计:")
        if IN_JUPYTER:
            display(df[numeric_cols].describe())
        else:
            print(df[numeric_cols].describe().to_string())

        # 可视化分布
        n_cols = 4
        n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows*4))
        axes = axes.flatten()

        for idx, col in enumerate(numeric_cols[:len(axes)]):
            ax = axes[idx]
            df[col].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
            ax.set_title(f'{col}\n范围: [{df[col].min():.2f}, {df[col].max():.2f}]')
            ax.set_xlabel('值')
            ax.set_ylabel('频数')

        # 隐藏多余的子图
        for idx in range(len(numeric_cols), len(axes)):
            axes[idx].set_visible(False)

        plt.suptitle('数值型特征分布', fontsize=16, y=1.02)
        plt.tight_layout()
        plt.show()

    else:
        print("❌ 没有数值型特征")

# %% [markdown]
# ## 6. 分类特征分析

# %%
if df is not None:
    # 提取分类列
    categorical_cols = df.select_dtypes(include=['object']).columns

    if len(categorical_cols) > 0:
        print(f"🏷️  找到 {len(categorical_cols)} 个分类特征")

        # 显示每个分类特征的唯一值数量
        print("\n🔍 分类特征唯一值统计:")
        unique_counts = {}
        for col in categorical_cols:
            unique_counts[col] = df[col].nunique()

        unique_df = pd.DataFrame({
            '特征': list(unique_counts.keys()),
            '唯一值数量': list(unique_counts.values())
        }).sort_values('唯一值数量', ascending=False)

        if IN_JUPYTER:
            display(unique_df)
        else:
            print(unique_df.to_string(index=False))

        # 可视化前几个分类特征
        top_n = min(6, len(categorical_cols))
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()

        for idx, col in enumerate(categorical_cols[:top_n]):
            ax = axes[idx]
            value_counts = df[col].value_counts()

            # 如果唯一值太多，只显示前10个
            if len(value_counts) > 10:
                top_values = value_counts.head(10)
                top_values.plot(kind='bar', ax=ax, alpha=0.7)
                ax.set_title(f'{col} (前10个)')
            else:
                value_counts.plot(kind='bar', ax=ax, alpha=0.7)
                ax.set_title(col)

            ax.set_xlabel('类别')
            ax.set_ylabel('计数')
            ax.tick_params(axis='x', rotation=45)

        # 隐藏多余的子图
        for idx in range(top_n, len(axes)):
            axes[idx].set_visible(False)

        plt.suptitle('分类特征分布', fontsize=16, y=1.02)
        plt.tight_layout()
        plt.show()

    else:
        print("❌ 没有分类特征")

# %% [markdown]
# ## 7. 相关性分析

# %%
if df is not None:
    # 选择数值型特征进行相关性分析
    numeric_df = df.select_dtypes(include=[np.number])

    if len(numeric_df.columns) > 1:
        print("🔗 计算特征相关性...")

        # 计算相关系数矩阵
        corr_matrix = numeric_df.corr()

        # 可视化相关系数矩阵
        plt.figure(figsize=(12, 10))

        # 创建热图
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        cmap = sns.diverging_palette(230, 20, as_cmap=True)

        sns.heatmap(corr_matrix, mask=mask, cmap=cmap, center=0,
                    square=True, linewidths=.5, cbar_kws={"shrink": .5},
                    annot=False, fmt=".2f")

        plt.title('特征相关系数矩阵', fontsize=16)
        plt.tight_layout()
        plt.show()

        # 找出相关性最强的特征对
        print("\n🔥 最强正相关特征对（前5）:")
        # 获取下三角矩阵（不包括对角线）
        corr_series = corr_matrix.unstack()
        corr_series = corr_series[corr_series.index.get_level_values(0) != corr_series.index.get_level_values(1)]
        corr_series = corr_series.sort_values(ascending=False)

        top_positive = corr_series.head(5)
        for (feat1, feat2), corr in top_positive.items():
            print(f"  {feat1} 和 {feat2}: {corr:.3f}")

        print("\n❄️  最强负相关特征对（前5）:")
        top_negative = corr_series.tail(5)[::-1]
        for (feat1, feat2), corr in top_negative.items():
            print(f"  {feat1} 和 {feat2}: {corr:.3f}")

    else:
        print("❌ 数值型特征不足，无法进行相关性分析")

# %% [markdown]
# ## 8. 目标变量分析

# %%
if df is not None:
    # 尝试识别目标变量
    target_candidates = ['GRADE', 'grade', 'Grade', 'GPA', 'gpa', '成绩', '分数']
    target_column = None

    for col in df.columns:
        if any(candidate in str(col).lower() for candidate in [t.lower() for t in target_candidates]):
            target_column = col
            break

    if target_column is None:
        # 尝试推断：可能是最后一列，或者数值型列中方差最大的
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            # 选择方差最大的数值列作为目标
            variances = df[numeric_cols].var()
            target_column = variances.idxmax()
            print(f"🤔 未找到明确的目标列，推断为: {target_column} (方差最大)")
        else:
            print("❌ 无法识别目标变量")
            target_column = None

    if target_column:
        print(f"🎯 目标变量: {target_column}")

        # 目标变量分布
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        df[target_column].hist(bins=30, edgecolor='black', alpha=0.7)
        plt.title(f'{target_column} 分布')
        plt.xlabel('值')
        plt.ylabel('频数')

        plt.subplot(1, 2, 2)
        plt.boxplot(df[target_column].dropna())
        plt.title(f'{target_column} 箱线图')
        plt.ylabel('值')

        plt.tight_layout()
        plt.show()

        # 统计信息
        print(f"\n📊 {target_column} 统计信息:")
        print(f"  平均值: {df[target_column].mean():.2f}")
        print(f"  标准差: {df[target_column].std():.2f}")
        print(f"  最小值: {df[target_column].min():.2f}")
        print(f"  最大值: {df[target_column].max():.2f}")
        print(f"  中位数: {df[target_column].median():.2f}")

        # 检查目标变量与其他特征的相关性
        numeric_features = df.select_dtypes(include=[np.number]).columns
        numeric_features = [f for f in numeric_features if f != target_column]

        if len(numeric_features) > 0:
            print(f"\n🔗 {target_column} 与其他特征的相关性（前10）:")
            correlations = {}
            for feature in numeric_features:
                corr = df[feature].corr(df[target_column])
                correlations[feature] = corr

            # 排序并显示
            corr_series = pd.Series(correlations)
            corr_series = corr_series.sort_values(ascending=False)

            # 显示前10个
            print("\n最正相关特征:")
            for feature, corr in corr_series.head(5).items():
                print(f"  {feature}: {corr:.3f}")

            print("\n最负相关特征:")
            for feature, corr in corr_series.tail(5)[::-1].items():
                print(f"  {feature}: {corr:.3f}")

# %% [markdown]
# ## 9. 数据质量问题检查

# %%
if df is not None:
    print("🔍 数据质量检查:")

    issues = []

    # 1. 检查重复行
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        issues.append(f"发现 {duplicates} 个重复行")

    # 2. 检查常数列
    constant_cols = []
    for col in df.columns:
        if df[col].nunique() == 1:
            constant_cols.append(col)

    if constant_cols:
        issues.append(f"发现常数列: {constant_cols}")

    # 3. 检查高缺失比例列
    missing_threshold = 0.5  # 50%
    high_missing_cols = []
    for col in df.columns:
        missing_ratio = df[col].isnull().mean()
        if missing_ratio > missing_threshold:
            high_missing_cols.append((col, missing_ratio))

    if high_missing_cols:
        issues.append(f"发现高缺失比例列（>50%）: {[col for col, _ in high_missing_cols]}")

    # 4. 检查数据类型一致性
    dtype_issues = []
    for col in df.columns:
        try:
            # 尝试转换为数值
            pd.to_numeric(df[col], errors='raise')
        except:
            # 如果失败，检查是否混合类型
            numeric_count = df[col].apply(lambda x: isinstance(x, (int, float))).sum()
            if 0 < numeric_count < len(df[col]):
                dtype_issues.append(col)

    if dtype_issues:
        issues.append(f"发现混合数据类型列: {dtype_issues}")

    # 输出检查结果
    if issues:
        print("⚠️  发现以下数据质量问题:")
        for i, issue in enumerate(issues, 1):
            print(f"  {i}. {issue}")
    else:
        print("✅ 未发现明显的数据质量问题")

# %% [markdown]
# ## 10. 总结与建议

# %%
print("📋 数据探索总结:")
print("=" * 50)

if df is not None:
    print(f"1. 数据规模: {df.shape[0]} 行 × {df.shape[1]} 列")
    print(f"2. 目标变量: {target_column if target_column else '未识别'}")

    numeric_cols = df.select_dtypes(include=[np.number]).columns
    categorical_cols = df.select_dtypes(include=['object']).columns

    print(f"3. 数值型特征: {len(numeric_cols)} 个")
    print(f"4. 分类特征: {len(categorical_cols)} 个")

    missing_total = df.isnull().sum().sum()
    if missing_total > 0:
        print(f"5. 缺失值: {missing_total} 个")
    else:
        print("5. 缺失值: 无")

    duplicates = df.duplicated().sum()
    if duplicates > 0:
        print(f"6. 重复行: {duplicates} 个")
    else:
        print("6. 重复行: 无")

print("\n🎯 下一步建议:")
print("1. 处理缺失值（如果需要）")
print("2. 处理异常值（如果需要）")
print("3. 编码分类特征（如果需要）")
print("4. 特征标准化/归一化")
print("5. 划分训练集和测试集")
print("6. 开始模型训练")

# %% [markdown]
# ## 11. 保存探索结果

# %%
# 可以在这里保存处理后的数据或分析结果
if df is not None and target_column:
    # 创建一个简单的预处理版本
    df_clean = df.copy()

    # 填充数值列的缺失值
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

    # 填充分类列的缺失值
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0] if len(df_clean[col].mode()) > 0 else 'Unknown')

    # 保存清理后的数据
    output_path = "../data/cleaned_data.csv"
    df_clean.to_csv(output_path, index=False)
    print(f"💾 清理后的数据已保存到: {output_path}")